## KLa Timeseries Generator

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import os

def generate_time_sequence(length):
    return [i / 96 for i in range(length)]

def create_nominal_kla_sequence(days, kla_value=120, tank=3):
    total_steps = days * 96
    if tank == 5:
        kla_value = kla_value / 2
    return [kla_value] * total_steps

def create_experiment_kla_sequence(days, nominal_kla, kla_value, DR_len, tank=3):
    steps_per_day = 96
    n_ininominal = 32
    n_DR = DR_len * 4
    n_fnlnominal = steps_per_day - n_ininominal - n_DR
    if tank == 5:
        nominal_kla = nominal_kla / 2
    day_pattern = [nominal_kla] * n_ininominal + [kla_value] * n_DR + [nominal_kla] * n_fnlnominal
    return day_pattern * days

def save_kla_to_mat(kla_sequence, tank, tag):
    time_seq = generate_time_sequence(len(kla_sequence))
    df = pd.DataFrame({'Sequence': time_seq, 'Value': kla_sequence})
    combined = df[['Sequence', 'Value']].values.tolist()

    var_name = f'KLa{tank}_Setpoints_ASM2'
    filename = f'BSM1-BSM2_MATLAB/BSM2_R2019b/KLa{tank}_Setpoints_BSM2_{tag}.mat'
    savemat(filename, {var_name: combined})
    print(f"Saved: {filename}")



In [ ]:
days = 609
kla_nominal_value = 120
kla_experiment_value = 0
experiment_length = 4  # hours

for tank in [3, 4, 5]:
    # Generate sequences
    nominal_seq = create_nominal_kla_sequence(days, kla_nominal_value, tank=tank)
    experiment_seq = create_experiment_kla_sequence(days, kla_nominal_value, kla_experiment_value, experiment_length, tank=tank)

    # Save to .mat files
    save_kla_to_mat(nominal_seq, tank=tank, tag='nominal')
    save_kla_to_mat(experiment_seq, tank=tank, tag='experiment')


Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa3_Setpoints_BSM2_nominal.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa3_Setpoints_BSM2_experiment.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa4_Setpoints_BSM2_nominal.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa4_Setpoints_BSM2_experiment.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa5_Setpoints_BSM2_nominal.mat
Saved: ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa5_Setpoints_BSM2_experiment.mat


## Old Python Iterator Script Below

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine

def create_segment(DR_len, kla, shift):
    n_calibration = (245 + shift*14)*96
    initial_segment = [120] * n_calibration
    
    n_ininominal = 32 #22 is roughly 8:00AM
    n_DR = DR_len * 4 
    n_fnlnominal = 96 - n_ininominal - n_DR
    
    segment = [120] * n_ininominal + [kla] * n_DR + [120] * n_fnlnominal
    final_segment = segment * 14
    return initial_segment + final_segment

def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= length:
            value = 0
    return seq

def save_to_mat(data):
    df = pd.DataFrame(data, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))
 
    # Create combined 2D list
    combined_data = df[['Sequence', 'Value']].values.tolist()
    
    # Save this combined list as a single variable in the .mat file
    data_dict_tank3 = {'KLa3_Setpoints_BSM2': combined_data}
    data_dict_tank4 = {'KLa4_Setpoints_BSM2': combined_data}
    data_dict_tank5 = {'KLa5_Setpoints_BSM2': combined_data}
    
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2.mat', data_dict_tank5)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2.mat', data_dict_tank4)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2.mat', data_dict_tank3)

def run_matlab_model(iteration):
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    iteration_data = {"iteration": np.array([iteration], dtype=np.float64)}
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_TSaeration_DRbsm2.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    data = create_segment(5,0,shift)
    save_to_mat(data)
    run_matlab_model(i + 1)  # Pass the current iteration count, starting from 1
    shift += 1



In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine

def run_DR_calibration_model():
    eng = matlab.engine.start_matlab()

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('DR_BSM2_calibration.m', nargout=0)
    eng.quit()

In [9]:
run_DR_calibration_model()

 
Running BSM2 to steady state! Solver = ode15s and Simulink model = benchmark2 ss
**************************************************************************
 
Steady state achieved. Initializing all state variables to steady state values.
 
Simulating BSM2 with dynamic influent (i) in open loop (Tempmodel = 1)! Solver = ode45 and Simulink model = benchmark2 open loop
*****************************************************************************************************************
 
Start time for simulation (hour:min:sec) = 20   3   2
Simulation paused at t = 14.0113 days.
Simulation resumed after custom actions.
the MATLAB function has been cancelled


Operation terminated by user during DR_BSM2_calibration


In run (line 91)
evalin('caller', strcat(script, ';'));

